In [0]:
catalog = "neobank"
vol = "/Volumes/neobank/neobank/neobank"
for t in ["customers", "accounts", "transactions", "branches", "credit_bureau_reports","payment_gateway_logs"]:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.{t}")
    dbutils.fs.rm(f"{vol}/_autoloader/checkpoints/{t}", True)


In [0]:
from pyspark.sql import functions as F

catalog = "neobank"
vol = "/Volumes/neobank/neobank/neobank"
state = f"{vol}/_autoloader"

dbutils.fs.mkdirs(f"{state}/schemas")
dbutils.fs.mkdirs(f"{state}/checkpoints")


def ingest_to_bronze(table_name, source_path, bronze_schema):
    target = f"{catalog}.{bronze_schema}.{table_name}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")

    before = spark.table(target).count() if spark.catalog.tableExists(target) else 0

    stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{state}/schemas/{table_name}")
        .load(source_path)
        .select("*", F.current_timestamp().alias("insert_timestamp"))
    )

    query = (
        stream.writeStream.format("delta")
        .option("checkpointLocation", f"{state}/checkpoints/{table_name}")
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(target)
    )
    query.awaitTermination()

    after = spark.table(target).count()
    return after - before

In [0]:
rows = (
    spark.table(f"{catalog}.metadata.tables")
    .filter("source_system = 'blob' AND active_flag = true")
    .orderBy("load_order")
    .collect()
)

for r in rows:
    n = ingest_to_bronze(r.table_name, r.source_path, r.bronze_schema)
    print(f"{r.table_name}: {n} new rows")

In [0]:
for t in [
    "customers",
    "accounts",
    "transactions",
    "credit_bureau_reports",
    "payment_gateway_logs",
    "branches",
]:
    print("=" * 30, t)
    spark.table(f"{catalog}.bronze.{t}").printSchema()

In [0]:
for t in [
    "customers",
    "accounts",
    "transactions",
    "branches",
    "credit_bureau_reports",
    "payment_gateway_logs",
]:
    df = spark.table(f"neobank.bronze.{t}")
    print(t, df.count(), "rescued:", df.filter("_rescued_data IS NOT NULL").count())